# DX 704 Week 6 Project

This project will develop a treatment plan for a fictious illness "Twizzleflu".
Twizzleflu is a mild illness caused by a virus.
The main symptoms are a mild fever, fidgeting, and kicking the blankets off the bed or couch.
Mild dehydration has also been reported in more severe cases.
These symptoms typically last 1-2 weeks without treatment.
Word on the internet says that Twizzleflu can be cured faster by drinking copious orange juice, but this has not been supported by evidence so far.
You will be provided with a theoretical model of Twizzleflu modeled as a Markov decision process.
Based on the model, you will compute optimal treatment plans to optimize different criteria, and compare patient discomfort with the different plans.

The full project description, a template notebook, and raw data are available on GitHub: [Project 6 Materials](https://github.com/bu-cds-dx704/dx704-project-06).

We will model Twizzleflu as a Markov decision process.
The model transition probabilities are provided in the file "twizzleflu-transitions.tsv" and the expected rewards are in "twizzleflu-rewards.tsv".
The goal for Twizzleflu is to minimize the expected discomfort of the patient which is expressed as negative rewards in the file.

## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Part 1: Evaluate a Do Nothing Plan

One of the treatment actions is to do nothing.
Calculate the expected discomfort (not rewards) of a policy that always does nothing.

Hint: for this value calculation and later ones, use value iteration.
The analytical solution has difficulties in practice when there is no discount factor.

In [2]:
import pandas as pd
import numpy as np

# load data
transitions = pd.read_csv("twizzleflu-transitions.tsv", sep="\t")
rewards = pd.read_csv("twizzleflu-rewards.tsv", sep="\t")

# states
states = sorted(rewards["state"].unique())

# build reward dict
R = dict(zip(rewards["state"], rewards["reward"]))

# filter to do-nothing action
T = transitions[transitions["action"] == "do_nothing"]

# initialize values
V = {s: 0.0 for s in states}

# value iteration / policy eval
theta = 1e-8
while True:
    delta = 0
    V_new = V.copy()

    for s in states:
        total = 0.0
        rows = T[T["state"] == s]

        for _, r in rows.iterrows():
            s_next = r["next_state"]
            p = r["prob"]
            total += p * V[s_next]

        V_new[s] = R[s] + total
        delta = max(delta, abs(V_new[s] - V[s]))

    V = V_new
    if delta < theta:
        break

Save the expected discomfort by state to a file "do-nothing-discomfort.tsv" with columns state and expected_discomfort.

In [3]:
out = pd.DataFrame({
    "state": list(V.keys()),
    "expected_discomfort": [-V[s] for s in V]
})

out.to_csv("do-nothing-discomfort.tsv", sep="\t", index=False)
out

,state,expected_discomfort
0,exposed-1,-0.0
1,exposed-2,-0.0
2,exposed-3,-0.0
3,recovered,-0.0
4,symptoms-1,1.0
5,symptoms-2,2.0
6,symptoms-3,1.0


Submit "do-nothing-discomfort.tsv" in Gradescope.

## Part 2: Compute an Optimal Treatment Plan

Compute an optimal treatment plan for Twizzleflu.
It should minimize the expected discomfort (maximize the rewards).

In [ ]:
transitions = pd.read_csv("twizzleflu-transitions.tsv", sep="\t")
rewards = pd.read_csv("twizzleflu-rewards.tsv", sep="\t")

# states
states = sorted(transitions["state"].unique())

# reward lookup: (state, action) -> reward
R = {(row["state"], row["action"]): row["reward"] for _, row in rewards.iterrows()}

# actions available per state
actions_by_state = transitions.groupby("state")["action"].unique().to_dict()

# transition lookup (state, action) -> list of (next_state, probability)
T = {}
for (s, a), df in transitions.groupby(["state", "action"]):
    T[(s, a)] = list(zip(df["next_state"].values, df["probability"].values))

# Value iteration for optimal control (gamma=1)
V = {s: 0.0 for s in states}
theta = 1e-10

while True:
    delta = 0.0
    V_new = V.copy()

    for s in states:
        best = -float("inf")

        for a in actions_by_state.get(s, []):
            exp_future = 0.0
            for s_next, p in T.get((s, a), []):
                exp_future += p * V[s_next]

            r = R.get((s, a), 0.0)
            q = r + exp_future  # gamma=1 here
            if q > best:
                best = q

        V_new[s] = best if best != -float("inf") else 0.0
        delta = max(delta, abs(V_new[s] - V[s]))

    V = V_new
    if delta < theta:
        break

Save the optimal actions for each state to a file "minimum-discomfort-actions.tsv" with columns state and action.

In [6]:
policy = {}
for s in states:
    best_a = None
    best_q = -float("inf")

    for a in actions_by_state.get(s, []):
        exp_future = 0.0
        for s_next, p in T.get((s, a), []):
            exp_future += p * V[s_next]

        r = R.get((s, a), 0.0)
        q = r + exp_future
        if q > best_q:
            best_q = q
            best_a = a

    policy[s] = best_a if best_a is not None else ""

out = pd.DataFrame({"state": list(policy.keys()), "action": list(policy.values())})
out.to_csv("minimum-discomfort-actions.tsv", sep="\t", index=False)
out

,state,action
0,exposed-1,sleep-8
1,exposed-2,sleep-8
2,exposed-3,sleep-8
3,recovered,do-nothing
4,symptoms-1,drink-oj
5,symptoms-2,drink-oj
6,symptoms-3,drink-oj


Submit "minimum-discomfort-actions.tsv" in Gradescope.

## Part 3: Expected Discomfort

Using your previous optimal policy, compute the expected discomfort for each state.

In [7]:
policy_df = pd.read_csv("minimum-discomfort-actions.tsv", sep="\t")

# policy: state -> action
policy = dict(zip(policy_df["state"], policy_df["action"]))

states = sorted(transitions["state"].unique())

# rewards
R = {(row["state"], row["action"]): row["reward"] for _, row in rewards.iterrows()}

# transitions
T = {}
for (s, a), df in transitions.groupby(["state", "action"]):
    T[(s, a)] = list(zip(df["next_state"].values, df["probability"].values))

Save your results in a file "minimum-discomfort-values.tsv" with columns state and expected_discomfort.

In [8]:
out = pd.DataFrame({
    "state": list(V.keys()),
    "expected_discomfort": [-V[s] for s in V]  # discomfort = -reward value
}).sort_values("state")

out.to_csv("minimum-discomfort-values.tsv", sep="\t", index=False)
out

,state,expected_discomfort
0,exposed-1,0.75
1,exposed-2,1.50
2,exposed-3,3.00
3,recovered,-0.00
4,symptoms-1,6.00
5,symptoms-2,4.50
6,symptoms-3,1.50


Submit "minimum-discomfort-values.tsv" in Gradescope.

## Part 4: Minimizing Twizzleflu Duration

Modifiy the Markov decision process to minimize the days until the Twizzle flu is over.
To do so, change the reward function to always be -1 if the current state corresponds to being sick (must have symptoms, exposed does not count) and 0 otherwise.
To be clear, the action does not matter for this reward function.


In [9]:
states = sorted(transitions["state"].unique())
actions = sorted(transitions["action"].unique())

def duration_reward(state: str) -> float:
    # sick means "has symptoms", exposed does NOT count
    return -1.0 if state.startswith("symptoms") else 0.0

# create a reward row for every (state, action) pair
duration_rewards = pd.DataFrame(
    [(a, s, duration_reward(s)) for s in states for a in actions],
    columns=["action", "state", "reward"]
)

duration_rewards

,action,state,reward
0,do-nothing,exposed-1,0.0
1,drink-oj,exposed-1,0.0
2,sleep-8,exposed-1,0.0
3,do-nothing,exposed-2,0.0
4,drink-oj,exposed-2,0.0
5,sleep-8,exposed-2,0.0
6,do-nothing,exposed-3,0.0
7,drink-oj,exposed-3,0.0
8,sleep-8,exposed-3,0.0
9,do-nothing,recovered,0.0


Save your new reward function in a file "duration-rewards.tsv" in the same format as "twizzleflu-rewards.tsv".

In [13]:
duration_rewards.to_csv("duration-rewards.tsv", sep="\t", index=False)

Submit "duration-rewards.tsv" in Gradescope.

## Part 5: Optimize for Shorter Twizzleflu

Compute an optimal policy to minimize the duration of Twizzleflu.

In [11]:
# actions available per state
actions_by_state = transitions.groupby("state")["action"].unique().to_dict()

# transition lookup
T = {}
for (s, a), df in transitions.groupby(["state", "action"]):
    T[(s, a)] = list(zip(df["next_state"].values, df["probability"].values))

# new reward: -1 if symptomatic, else 0 (action-independent)
def R_duration(s: str) -> float:
    return -1.0 if str(s).startswith("symptoms") else 0.0

# value iteration for optimal control (gamma=1)
V = {s: 0.0 for s in states}
theta = 1e-10

while True:
    delta = 0.0
    V_new = V.copy()

    for s in states:
        best = -float("inf")
        r = R_duration(s)

        for a in actions_by_state.get(s, []):
            exp_future = 0.0
            for s_next, p in T.get((s, a), []):
                exp_future += p * V[s_next]

            q = r + exp_future  # gamma=1
            if q > best:
                best = q

        V_new[s] = best if best != -float("inf") else r
        delta = max(delta, abs(V_new[s] - V[s]))

    V = V_new
    if delta < theta:
        break

Save the optimal actions for each state to a file "minimum-duration-actions.tsv" with columns state and action.

In [12]:
# greedy optimal policy - save
policy = {}
for s in states:
    best_a = None
    best_q = -float("inf")
    r = R_duration(s)

    for a in actions_by_state.get(s, []):
        exp_future = 0.0
        for s_next, p in T.get((s, a), []):
            exp_future += p * V[s_next]

        q = r + exp_future
        if q > best_q:
            best_q = q
            best_a = a

    policy[s] = best_a if best_a is not None else ""

out = pd.DataFrame({"state": list(policy.keys()), "action": list(policy.values())})
out.to_csv("minimum-duration-actions.tsv", sep="\t", index=False)
out

,state,action
0,exposed-1,sleep-8
1,exposed-2,sleep-8
2,exposed-3,sleep-8
3,recovered,do-nothing
4,symptoms-1,do-nothing
5,symptoms-2,do-nothing
6,symptoms-3,do-nothing


Submit "minimum-duration-actions.tsv" in Gradescope.

## Part 6: Shorter Twizzleflu?

Compute the expected number of days sick for each state to a file.

In [15]:
transitions = pd.read_csv("twizzleflu-transitions.tsv", sep="\t")
policy_df = pd.read_csv("minimum-duration-actions.tsv", sep="\t")  # from Part 5

policy = dict(zip(policy_df["state"], policy_df["action"]))
states = sorted(transitions["state"].unique())

# transitions
T = {}
for (s, a), df in transitions.groupby(["state", "action"]):
    T[(s, a)] = list(zip(df["next_state"].values, df["probability"].values))

def r_duration(s: str) -> float:
    return -1.0 if str(s).startswith("symptoms") else 0.0

# policy evaluation via iterative updates (gamma=1)
V = {s: 0.0 for s in states}
theta = 1e-10

while True:
    delta = 0.0
    V_new = V.copy()

    for s in states:
        a = policy.get(s, "")
        exp_future = 0.0
        for s_next, p in T.get((s, a), []):
            exp_future += p * V[s_next]

        V_new[s] = r_duration(s) + exp_future
        delta = max(delta, abs(V_new[s] - V[s]))

    V = V_new
    if delta < theta:
        break

Save the expected sick days for each state to a file "minimum-duration-days.tsv" with columns state and expected_sick_days.

In [16]:
# expected sick days = - expected total reward
out = pd.DataFrame({
    "state": list(V.keys()),
    "expected_days_sick": [-V[s] for s in V]
}).sort_values("state")

out.to_csv("minimum-duration-days.tsv", sep="\t", index=False)
out

,state,expected_days_sick
0,exposed-1,1.250000
1,exposed-2,2.500000
2,exposed-3,5.000000
3,recovered,-0.000000
4,symptoms-1,10.000000
5,symptoms-2,6.666667
6,symptoms-3,3.333333


Submit "minimum-duration-days.tsv" in Gradescope.

## Part 7: Speed vs Pampering

Compute the expected discomfort using the policy to minimize days sick, and compare the results to the expected discomfort when optimizing to minimize discomfort.

In [17]:
# load both policies
speed_pol_df = pd.read_csv("minimum-duration-actions.tsv", sep="\t")
min_dis_pol_df = pd.read_csv("minimum-discomfort-actions.tsv", sep="\t")

speed_policy = dict(zip(speed_pol_df["state"], speed_pol_df["action"]))
min_dis_policy = dict(zip(min_dis_pol_df["state"], min_dis_pol_df["action"]))

states = sorted(transitions["state"].unique())

# reward lookup
R = {(row["state"], row["action"]): row["reward"] for _, row in rewards.iterrows()}

# transition lookup
T = {}
for (s, a), df in transitions.groupby(["state", "action"]):
    T[(s, a)] = list(zip(df["next_state"].values, df["probability"].values))

def evaluate_policy(policy: dict, theta: float = 1e-10) -> dict:
    """Return V^pi(s) for all states using iterative policy evaluation (gamma=1)."""
    V = {s: 0.0 for s in states}

    while True:
        delta = 0.0
        V_new = V.copy()

        for s in states:
            a = policy.get(s, "")
            r = R.get((s, a), 0.0)

            exp_future = 0.0
            for s_next, p in T.get((s, a), []):
                exp_future += p * V[s_next]

            V_new[s] = r + exp_future  # gamma=1
            delta = max(delta, abs(V_new[s] - V[s]))

        V = V_new
        if delta < theta:
            break

    return V

# evaluate both policies under the ORIGINAL discomfort rewards
V_speed = evaluate_policy(speed_policy)
V_min_dis = evaluate_policy(min_dis_policy)

# cnvert to expected discomfort (discomfort = -value)
comparison = pd.DataFrame({
    "state": states,
    "speed_discomfort": [-V_speed[s] for s in states],
    "minimize_discomfort": [-V_min_dis[s] for s in states],
})

Save the results to a file "policy-comparison.tsv" with columns state, speed_discomfort, and minimize_discomfort.

In [18]:
comparison.to_csv("policy-comparison.tsv", sep="\t", index=False)
comparison

,state,speed_discomfort,minimize_discomfort
0,exposed-1,0.833333,0.75
1,exposed-2,1.666667,1.50
2,exposed-3,3.333333,3.00
3,recovered,-0.000000,-0.00
4,symptoms-1,6.666667,6.00
5,symptoms-2,5.000000,4.50
6,symptoms-3,1.666667,1.50


Submit "policy-comparison.tsv" in Gradescope.

## Part 8: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.

## Part 9: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.

In [19]:
content = (
    "Acknowledgments\n\n"
    "I used the DX704 course-provided code examples as a reference when developing "
    "the code for this assignment.\n\n"
    "I referenced the following notebook from the course repo:\n"
    "https://github.com/bu-cds-omds/dx704-examples/blob/main/week01/"
    "chart_risk_return_performance_of_efficient_portfolios.ipynb\n"
    "I also used reddit and youtube.\n"
)

with open("acknowledgments.txt", "w") as f:
    f.write(content)